###1. Automated Support Triage & Ticket Routing
Customer service teams are often overwhelmed with a massive backlog of tickets. Instead of having humans read every ticket to decide who handles it, your pipeline does it instantly.

###NLP Techniques (Syntax & Structure)
Tokenization: Splits text into individual words and punctuation.
POS Tagging: Assigns grammatical labels (nouns, verbs, adjectives).
Vector Embeddings: Converts text meaning into mathematical numbers.
NLU Techniques (Meaning & Context)
Named Entity Recognition (NER): Extracts specific real-world subjects (products, locations).
Sentiment Analysis: Identifies emotional tone (Positive, Negative, Neutral).
Intent Detection: Determines the user's specific goal (returns, complaints).
How to create Databricks Token
In your Databricks workspace, click your username in the top bar and select Settings.
Click Developer.
Next to Access tokens, click Manage.
Click Generate new token.

In [0]:
#databricks secrets create-scope izgenaiscope
#databricks secrets put-secret izgenaiscope databricks_token
DATABRICKS_TOKEN = dbutils.secrets.get(scope="izgenaiscope", key="databricks_token")

  File <command-5803466066311690>, line 1
    databricks secrets create-scope izgenaiscope
               ^
SyntaxError: invalid syntax


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakehousecat;
CREATE SCHEMA IF NOT EXISTS lakehousecat.default;
drop table if exists lakehousecat.default.customer_reviews;
CREATE TABLE IF NOT EXISTS lakehousecat.default.customer_reviews (
  review_id STRING,
  city string,
  review_text STRING);
INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-001",'NYC', "The shoe are beautiful, but they ripped after two days of wearing them! I want my money back."),
("REV-002",'NY', "The Laptop Shipping took 3 weeks. Absolutely unacceptable."),
("REV-003",'CA', "Perfect fit! Will definitely be buying from you guys again."),
("REV-004",'CAL', "I received the wrong color. I ordered black but got blue. How do I fix this?");

INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-005",'NYC', "Do you know how much mark my son is going to get in this exam?");

INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-006",'NYC', "phone purchased a week ago for $500 in NewYork");
     

num_affected_rows,num_inserted_rows
1,1


In [0]:
df_reviews = spark.read.table("lakehousecat.default.customer_reviews")
display(df_reviews)

review_id,city,review_text
REV-001,NYC,"The shoe are beautiful, but they ripped after two days of wearing them! I want my money back."
REV-002,NY,The Laptop Shipping took 3 weeks. Absolutely unacceptable.
REV-003,CA,Perfect fit! Will definitely be buying from you guys again.
REV-004,CAL,I received the wrong color. I ordered black but got blue. How do I fix this?
REV-005,NYC,Do you know how much mark my son is going to get in this exam?
REV-006,NYC,phone purchased a week ago for in NewYork


###NLP & NLU using Text Generation Model
here ai_query is a function.. which takes model name and prompt as i/p and return the output in a string? Is it a spark function or databricks func?

In [0]:
%sql
CREATE OR REPLACE VIEW lakehousecat.default.advanced_text_analysis AS
SELECT 
  review_id,
  review_text,

  -- NLP CONCEPTS: Syntax, Structure, & Mechanics
--Prompt engineering techniques we used here - Major 2 types of prompts - System & User Prompt (Role Prompt, Instruction Prompt, Output Prompt.)
  -- 1. Tokenization: Breaking text into individual pieces
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a tokenizer. Break the following text down into individual words and punctuation marks. Return ONLY a comma-separated list of tokens: ', review_text)
  ) AS nlp_tokens,

  -- 2. POS (Part-of-Speech) Tagging: Identifying grammatical labels
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a POS tagger. Identify the part of speech for the words in this text. Return ONLY a comma-separated list in "word(TAG)" format (e.g., fast(ADJECTIVE), run(VERB)): ', review_text)
  ) AS nlp_pos_tags,

  -- 3. Vector Embeddings: Converting tex into mathematical (number) representations
  -- Note: Text generation models (like Llama) don't create embeddings. 
  ai_query(
    'databricks-bge-large-en', 
    review_text) AS nlp_vector_embedded_values,

--The below rephrased words is not the part of NLP operation, just to show GenAI generates rather than use the existing code.
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a rephraser. From the given text, can you rephrase the entire text in 2 other ways. Return rephrased sentence in double quotes terminated by comma: ', review_text)
  ) AS example_of_GenAI_rephrased_words,

  -- NLU CONCEPTS: Meaning, Intent, Sentiment & Context
  --NLU 2 major roles - 1. Intent Identification , 2. Entity Recognition.

  -- 4. NER (Named Entity Recognition): Extracting specific real-world subjects
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are an NER tool., Extract Named Entities such as Person, Organization, Duration, Time, Money, Location, or Product. Return ONLY a comma-separated list in "Entity (Type)" format. If none exist, return "None": ', review_text)
  ) AS nlu_ner_entities,

  -- 5. Semantic Matching (Similarity Scoring) : Checking for a specific underlying goal (This example is to show the similarity scoring used behind)
  --by converting intent & given prompt into vectors
  --product is not useful,money back [1.2,4.1]= refund [1.2,4]
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Does this review express an intent to return the product, ask for a refund, or complain about shipping duration? Return ONLY the word YES or NO: ', review_text)
  ) AS nlu_return_semantic_matching,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Information", "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_8b_intent,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Information", "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_70b_intent,
  
  -- 7. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify the primary intent & sentiment of this customer review. Choose ONLY ONE from the following department we have to route this intented and sentiment to customer request such as: "Logistics Department", "Support Department","Quality Control Department", "Escalation" or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_70b_intent_routing,

  -- 7. Sentiment Analysis: Understanding emotional tone
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a sentiment analyzer. Classify the emotional tone of this review, dont hallucinate. Return sentiment score in a range of -1 to 1: ', review_text)
  ) AS sentiment_scoring

FROM lakehousecat.default.customer_reviews;

In [0]:
%sql
SELECT * FROM lakehousecat.default.advanced_text_analysis;

review_id review_text nlp_tokens nlp_pos_tags nlp_vector_embedded_values example_of_GenAI_rephrased_words nlu_ner_entities nlu_return_semantic_matching nlu_primary_8b_intent nlu_primary_70b_intent nlu_primary_70b_intent_routing sentiment_scoring REV-001 The shoe are beautiful, but they ripped after two days of wearing them! I want my money back. Here is the list of tokens:

The, shoe, are, beautiful, but, they, ripped, after, two, days, of, wearing, them, I, want, my, money, back shoe(NOUN), are(VBZ), beautiful(ADJECTIVE), but(CONJUNCTION), they(PRONOUN), ripped(VERB), after(PREPOSITION), two(NUMBER), days(NOUN), of(PREPOSITION), wearing(VERB), them(PRONOUN), I(PRONOUN), want(VERB), my(PRONOUN), money(NOUN), back(NOUN) List(-0.01369476318359375, 0.0264892578125, 0.005191802978515625, 0.00254058837890625, -0.025146484375, 0.0423583984375, 0.042694091796875, 0.048583984375, 0.017486572265625, -0.004512786865234375, 0.030426025390625, 0.0264892578125, 0.0260162353515625, -0.033660888671875, -0.006381988525390625, -0.013916015625, 0.03582763671875, -0.058685302734375, -0.00661468505859375, 0.0016651153564453125, 0.00353240966796875, 0.009124755859375, -0.032196044921875, -0.007671356201171875, 0.01216888427734375, 0.02581787109375, 0.029144287109375, -0.0200042724609375, 0.031463623046875, 0.047271728515625, -0.052001953125, -0.040740966796875, 2.803802490234375E-4, -0.032989501953125, 0.0167999267578125, 0.001068115234375, 0.01910400390625, 0.0156402587890625, -0.02374267578125, -0.0204925537109375, -0.01435089111328125, -0.03619384765625, 0.035980224609375, -0.0592041015625, 0.0177154541015625, -0.005931854248046875, -0.016815185546875, -0.052398681640625, -0.01465606689453125, -0.043121337890625, 0.0380859375, -0.015777587890625, -0.0093536376953125, -0.01541900634765625, -0.002651214599609375, -0.0128936767578125, 0.007904052734375, -0.0104217529296875, -0.015228271484375, 0.034454345703125, 0.033538818359375, -0.017303466796875, 0.04730224609375, -0.06494140625, 0.0102386474609375, -0.05731201171875, -0.035125732421875, -0.0016574859619140625, 0.019927978515625, -0.014373779296875, 0.00984954833984375, 0.048004150390625, 0.01491546630859375, -0.006717681884765625, -0.01251220703125, 0.060638427734375, -0.02496337890625, -0.00511932373046875, 0.0250244140625, 0.0343017578125, 0.031097412109375, 2.47955322265625E-4, 0.0692138671875, 0.048828125, -0.0283203125, 0.007579803466796875, 0.004207611083984375, 0.004772186279296875, 0.00600433349609375, -0.0050506591796875, -0.04388427734375, 0.035614013671875, -0.01309967041015625, -0.0113067626953125, 0.049591064453125, 0.0241241455078125, -0.0267486572265625, -0.0080413818359375, -0.01459503173828125, 0.0101776123046875, 0.039398193359375, -0.020751953125, 0.0030612945556640625, 0.05426025390625, -0.06109619140625, -0.0033855438232421875, 0.0693359375, -0.0205230712890625, -0.00818634033203125, -0.036041259765625, -0.0028133392333984375, 0.0305328369140625, 0.06024169921875, 6.418228149414062E-4, -0.024810791015625, 0.00765228271484375, 0.01168060302734375, -0.002452850341796875, -0.04339599609375, -0.03704833984375, 0.0192108154296875, 0.044158935546875, -0.0012054443359375, 6.818771362304688E-4, 0.03948974609375, -0.039093017578125, -0.01146697998046875, 0.07476806640625, 0.01360321044921875, 0.00579833984375, 0.0079803466796875, -0.0113525390625, 0.0012798309326171875, -0.01425933837890625, 0.0118865966796875, -0.015899658203125, 0.008697509765625, -0.0159454345703125, 0.03753662109375, -0.0299835205078125, 0.03594970703125, 0.03729248046875, 0.0140533447265625, 0.08062744140625, -0.0240631103515625, 0.03887939453125, 0.07806396484375, 0.012115478515625, -0.0535888671875, -0.006938934326171875, -0.050537109375, -0.0176849365234375, -0.0139617919921875, 0.032196044921875, 0.003955841064453125, 0.0284423828125, -0.014068603515625, -0.003971099853515625, 0.006603240966796875, -0.01500701904296875, 0.02056884765625, 0.006595611572265625, -0.031951904296875, 0.046783447265625, -0.0192

In [0]:
%sql
select ai_query(
    'databricks-meta-llama-3-1-8b-instruct',concat('you are a doctor, you have to suggest a treatment for the following condition',review_text)) as treatment
     FROM lakehousecat.default.customer_reviews;
     

treatment
"I think there may be a bit of a misunderstanding here. As a doctor, I'm not sure I can provide a medical treatment for a ripped shoe. However, I can offer some advice on how to approach the situation. It sounds like you're experiencing a case of ""shoe disappointment"" or ""retail frustration."" Here are a few suggestions: 1. **Contact the store**: Reach out to the retailer where you purchased the shoes and explain the situation. They may be willing to offer a refund, exchange, or repair the shoes for you. 2. **Check the warranty**: If the shoes came with a warranty, review it to see if it covers defects or damage within a certain timeframe. 3. **Consider a repair**: If the shoes are still in good condition otherwise, you might consider taking them to a cobbler or shoe repair service to see if they can fix the rip. However, if you're experiencing emotional distress or anxiety related to the situation, I'd be happy to offer some general advice on stress management or coping strategies. Would you like to discuss this further?"
"A very... unusual condition! As a doctor, I'd like to diagnose this as a case of ""Shipping-Related Frustration Syndrome"" (SRFS). This condition is characterized by feelings of anger, disappointment, and frustration due to delayed or unacceptable shipping times. To treat SRFS, I recommend the following treatment plan: **Initial Treatment:** 1. **Calmative Therapy**: I prescribe a 10-minute break from the laptop and its shipping woes. Take a few deep breaths, and engage in a relaxing activity, such as meditation, reading, or a short walk. 2. **Communication Therapy**: Reach out to the shipping company's customer service department to express your concerns and frustration. This may help to resolve the issue and provide a sense of closure. 3. **Reframing Therapy**: Challenge your negative thoughts about the shipping company and the laptop's delivery time. Remind yourself that delays can happen, and it's not a reflection of your worth or the laptop's quality. **Follow-up Treatment:** 1. **Shipping Company Intervention**: If the shipping company is unresponsive or unhelpful, consider escalating the issue to a supervisor or a higher authority. 2. **Alternative Shipping Options**: Research and explore alternative shipping methods, such as expedited shipping or a different courier service, to ensure a faster delivery time. 3. **Laptop-Related Activities**: Engage in activities that bring you joy and distract you from the shipping woes, such as playing games, watching movies, or working on a different project. **Preventative Measures:** 1. **Shipping Company Research**: Research the shipping company's reputation, reviews, and ratings before using their services. 2. **Clear Communication**: Clearly communicate your expectations and deadlines to the shipping company to avoid misunderstandings. 3. **Backup Plans**: Develop a backup plan, such as having a spare laptop or a different shipping option, to minimize the impact of shipping delays. By following this treatment plan, you should be able to manage your SRFS symptoms and find a resolution to your laptop shipping issue."
"I think there may be a misunderstanding here. As a doctor, I'm not sure what condition you're referring to, as your message seems to be a positive review rather than a description of a medical issue. Could you please provide more information about the condition you're experiencing, such as symptoms, duration, and any relevant medical history? That way, I can provide a more accurate and helpful treatment suggestion."
"A rather...unconventional medical conundrum! While I'm happy to help, I must clarify that receiving the wrong color of a product is not a medical condition per se. However, I'll play along and offer a tongue-in-cheek treatment plan. **Diagnosis:** ""Chromatic Dissonance Syndrome"" (CDS) **Symptoms:** Patient reports feeling distressed, frustrated, and possibly even embarrassed due to the mismatch between the ordered and r

###2. Automated Customer Review Response Generator (E-Commerce)
Instead of just classifying the sentiments and intents, We can use LLM (NLG) to actually write the email response to the customer.

The Pipeline Concept: Read the negative review from a Delta table, pass it to the LLM with strict instructions, and output a drafted email into a new column.

LLM & GenAI: LLM is a Large Lang Model using Meta Llama LLM, we are going to generate the mail output using NLG (GenAI+LLM) based on the input review text.

NLG (Natural Language Generation): The model synthesizes a polite, grammatically perfect, and empathetic email (as per our prompt)

Grounding: Pass the company's official return policy into the system prompt. The AI is grounded in this specific document so it doesn't make up rules.

Guardrailing: Add a strict rule to the prompt: "Never promise a refund or discount. Only offer to connect them to the support team."

Hallucination (Anti) Mitigation: Set temperature=0.1, so the AI doesn't invent fake product features or invent a fake customer service rep name.

In [0]:
%sql
CREATE OR REPLACE TABLE lakehousecat.default.ecommerce_raw_reviews (
  review_id STRING,
  customer_review STRING,
  custname string
);

INSERT INTO lakehousecat.default.ecommerce_raw_reviews (review_id, customer_review,custname)
VALUES 
  ('REV-001', 'The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.','Irfan'),
  ('REV-002', 'Shipping took 3 weeks. Absolutely unacceptable.','Vaanmathy'),
  ('REV-003', 'Perfect fit! Will definitely be buying from you guys again.','Sarangabani'),
  ('REV-004', 'I received the wrong color. I ordered black but got blue. How do I fix this?','Vasu');

SELECT * FROM lakehousecat.default.ecommerce_raw_reviews;

review_id,customer_review,custname
REV-001,"The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.",Irfan
REV-002,Shipping took 3 weeks. Absolutely unacceptable.,Vaanmathy
REV-003,Perfect fit! Will definitely be buying from you guys again.,Sarangabani
REV-004,I received the wrong color. I ordered black but got blue. How do I fix this?,Vasu


In [0]:
%sql
--What we learn here - Prompt Engineering, Context Engineering, Grounding, Guardrailing, Anti Hallucination, Human In Loop, Model Parameters, NLG
SELECT 
    review_id,
    customer_review,
    custname,
    ai_query(
        -- 1. The Model Endpoint
        'databricks-meta-llama-3-3-70b-instruct',
        
        -- 2. The Contextual Prompt (System prompt/context (Grounding/Guardrailing/Anti Hallucination) + The Data Column (User Prompt))
        CONCAT(
            'You are an empathetic, professional customer support Engineer for an E-Commerce company. ',
            'Read the customer review and write a direct email response to them addressing the customer. ',
            'Produce the output with subject, salutation, body, closing message, and signature with - Thanks & Regards, Inceptez Technologies',
            
            'GROUNDING CONTEXT: ',
            '- We only accept returns within 30 days of purchase. ',
            '- We do NOT give cash refunds. We only offer store credit or exact item replacements. ',
            
            'GUARDRAILS: ',
            '1. Never promise a refund through electronic money transfer under any circumstances. ',
            '2. Never offer a discount code or coupon. ',
            '3. Keep the response under 4 sentences when you do NLG. ',
            
            'ANTI HALLUCINATION: ',
            'If the user asks for something not covered in the policies above, do not invent a solution. ',
            'Simply state: "I am escalating this to our senior support team who will contact you within 24 hours." ',
            
            'Customer Review: ', customer_review, custname
        ),
        
        -- 3. The Model Parameters (Ensuring robotic/strict adherence to guardrails)
        modelParameters => named_struct('temperature', 0, 'max_tokens', 100)
        
    ) AS ai_generated_email

FROM lakehousecat.default.ecommerce_raw_reviews;

review_id,customer_review,custname,ai_generated_email
REV-001,"The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.",Irfan,"Subject: Concern with Recent Shoe Purchase Dear Irfan, I apologize for the issue you've experienced with your recent shoe purchase, and I'm sorry to hear that they ripped after only two days of wear. As per our return policy, we can offer a store credit or an exact item replacement if you return the shoes within 30 days of purchase. I am escalating this to our senior support team who will contact you within 24 hours to discuss further. Thanks & Regards, Inceptez"
REV-002,Shipping took 3 weeks. Absolutely unacceptable.,Vaanmathy,"Subject: Concern with Shipping Time - Order Review Dear Vaanmathy, I apologize for the delay in shipping your order, which took 3 weeks to arrive, and I understand that this is unacceptable. I am escalating this to our senior support team who will contact you within 24 hours to discuss possible solutions, such as store credit or item replacement, as per our return policy. Thanks & Regards, Inceptez Technologies"
REV-003,Perfect fit! Will definitely be buying from you guys again.,Sarangabani,"Subject: Re: Positive Experience with Our Store Dear Sarangabani, We are thrilled to hear that our product was a perfect fit for you and that you're looking forward to making another purchase with us. We appreciate your loyalty and can't wait to serve you again. If you have any questions or need assistance in the future, please don't hesitate to reach out. Thanks & Regards, Inceptez Technologies"
REV-004,I received the wrong color. I ordered black but got blue. How do I fix this?,Vasu,"Subject: Return and Replacement Inquiry for Incorrect Item Dear Vasu, I apologize for the inconvenience you've experienced with receiving the wrong color. To resolve this issue, you can initiate a return within 30 days of purchase and we can offer a store credit or an exact item replacement in the correct color, black. Please let me know if you would prefer a store credit or a replacement, and I will guide you through the next steps. Thanks & Regards, Inceptez Technologies"
